# 1D CNN + FastText (PyTorch) 



In [ ]:
import os
import random
import numpy as np
import pandas as pd
import warnings
from collections import Counter
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)
import fasttext
import wandb

# ── Reproducibility ───────────────────────────────────────────────
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Libraries loaded.')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

In [ ]:
wandb_key = os.environ.get('WANDB_API_KEY')
if not wandb_key:
    raise ValueError('Set WANDB_API_KEY in your .env file')

wandb.login(key=wandb_key)
wandb.init(
    project = 'commitment-mining',
    name    = 'dl-simplecnn-fasttext',
    config  = {'model': 'SimpleCNN (single Conv1D branch)', 'embedding': 'FastText', 'framework': 'pytorch'},
    tags    = ['cnn', 'deep-learning', 'fasttext', 'pytorch', 'single-branch']
)
print('WandB initialized.')

In [ ]:
TRAIN_PATH = 'Commitment-Mining/dataset/train_80p.xlsx'
TEST_PATH  = 'Commitment-Mining/dataset/test_20p.xlsx'
TEXT_COL   = 'statements (ne)'
LABEL_COL  = 'final_label'

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

for df in [train_df, test_df]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

print(f'Train size : {len(train_df)}')
print(f'Test size  : {len(test_df)}')
print('\nTrain label distribution:')
print(train_df[LABEL_COL].value_counts())

label_counts = train_df[LABEL_COL].value_counts().to_dict()
wandb.log({'train_size': len(train_df), 'test_size': len(test_df), **label_counts})

In [ ]:
train_df['sentence_length'] = train_df[TEXT_COL].apply(
    lambda x: pd.cut(
        [len(x.split())],
        bins=[0, 5, 10, 20, 50, 999],
        labels=['xs', 's', 'm', 'l', 'xl']
    )[0]
)

train_df['strat_key'] = (
    train_df['province'].astype(str)           + '_' +
    train_df['sentence_length'].astype(str)    + '_' +
    train_df['district/gaupalika'].astype(str) + '_' +
    train_df[LABEL_COL].astype(str)
)

counts = train_df['strat_key'].value_counts()
rare   = counts[counts < 5].index
train_df['strat_key'] = train_df['strat_key'].apply(
    lambda x: 'rare' if x in rare else x
)

print(f'Unique strat keys : {train_df["strat_key"].nunique()}')

In [ ]:
le      = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL].values)
y_test  = le.transform(test_df[LABEL_COL].values)
groups  = train_df['strat_key'].values

X_train_text = [str(x) for x in train_df[TEXT_COL].tolist()]
X_test_text  = [str(x) for x in test_df[TEXT_COL].tolist()]

print(f'Classes  : {le.classes_}')
print(f'y_train  : {y_train.shape} | dtype: {y_train.dtype}')

## Domain-adapted FastText

Trained on `X_train_text` only (never on val/test text) to avoid leakage.

In [ ]:
import os
import fasttext

EMBED_DIM = 300
USE_DOMAIN_FASTTEXT = True          # domain-adapted (now pretrained-initialized) vs pure generic
FASTTEXT_VEC_PATH = '/home/rupak/Desktop/Commitment-Mining/embeddings/cc.ne.300.vec'
FASTTEXT_BIN_PATH = '/home/rupak/Desktop/Commitment-Mining/embeddings/cc.ne.300.bin'

if USE_DOMAIN_FASTTEXT:
    print('Training FastText on training data, initialized from pretrained cc.ne.300 vectors...')

    # Generate the .vec file from the .bin once, if it doesn't exist yet
    if not os.path.exists(FASTTEXT_VEC_PATH):
        print('  .vec not found — extracting it from .bin (one-time, may take a few minutes)...')
        pretrained_model = fasttext.load_model(FASTTEXT_BIN_PATH)
        words = pretrained_model.get_words()
        dim = pretrained_model.get_dimension()
        with open(FASTTEXT_VEC_PATH, 'w', encoding='utf-8') as f:
            f.write(f"{len(words)} {dim}\n")
            for w in words:
                vec_str = ' '.join(map(str, pretrained_model.get_word_vector(w)))
                f.write(f"{w} {vec_str}\n")
        del pretrained_model  # free memory before the next training step
        print(f'  Saved {FASTTEXT_VEC_PATH}')

    FASTTEXT_TRAIN_CORPUS = 'fasttext_train_corpus.txt'
    with open(FASTTEXT_TRAIN_CORPUS, 'w', encoding='utf-8') as f:
        for text in X_train_text:      # TRAIN TEXT ONLY -- no val/test leakage
            f.write(text + '\n')

    ft_model = fasttext.train_unsupervised(
        FASTTEXT_TRAIN_CORPUS,
        model='skipgram',
        dim=EMBED_DIM,
        epoch=20,
        lr=0.05,
        wordNgrams=2,
        minCount=3,
        ws=5,
        thread=4,
        pretrainedVectors=FASTTEXT_VEC_PATH   # <-- the actual change
    )
    ft_model.save_model('trained_fasttext_model_pytorch.bin')
    print('Pretrained-initialized, domain-fine-tuned FastText model trained and saved.')

    if os.path.exists(FASTTEXT_TRAIN_CORPUS):
        os.remove(FASTTEXT_TRAIN_CORPUS)
else:
    print('Loading generic pretrained FastText model (no domain fine-tuning)...')
    ft_model = fasttext.load_model(FASTTEXT_BIN_PATH)

print(f'FastText ready. Embedding dim: {EMBED_DIM}')

## Tokenizer, padding, embedding matrix

`MAX_LEN = 64` (per your spec; ~20 outlier sentences out of your dataset exceed this and get truncated). Full vocabulary kept (no cap).

In [ ]:
MAX_LEN   = 64
OOV_TOKEN = '<OOV>'
PAD_IDX   = 0
OOV_IDX   = 1

class SimpleTokenizer:
    def __init__(self, num_words=None, oov_token=OOV_TOKEN):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}

    def fit_on_texts(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(t.split())
        self.word_index[self.oov_token] = OOV_IDX
        most_common = counter.most_common() if self.num_words is None \
            else counter.most_common(self.num_words - 2)
        for i, (word, _) in enumerate(most_common):
            self.word_index[word] = i + 2  # 0=pad, 1=oov

    def texts_to_sequences(self, texts):
        return [[self.word_index.get(w, OOV_IDX) for w in t.split()] for t in texts]


def pad_sequences(sequences, maxlen, padding='post', truncating='post'):
    out = np.zeros((len(sequences), maxlen), dtype=np.int64)
    lengths = np.zeros(len(sequences), dtype=np.int64)
    for i, seq in enumerate(sequences):
        if len(seq) > maxlen:
            seq = seq[:maxlen] if truncating == 'post' else seq[-maxlen:]
        lengths[i] = max(len(seq), 1)
        if padding == 'post':
            out[i, :len(seq)] = seq
        else:
            out[i, maxlen - len(seq):] = seq
    return out, lengths


tokenizer = SimpleTokenizer(num_words=None, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_text)
vocab_size = len(tokenizer.word_index) + 1

X_train_seq, len_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding='post', truncating='post'
)
X_test_seq, len_test = pad_sequences(
    tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN, padding='post', truncating='post'
)

print(f'Vocab size (uncapped) : {vocab_size}')
print(f'X_train_seq    : {X_train_seq.shape}')
print(f'X_test_seq     : {X_test_seq.shape}')

embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype=np.float32)
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size:
        embedding_matrix[idx] = ft_model.get_word_vector(word)

print(f'Embedding matrix : {embedding_matrix.shape}')

wandb.log({'vocab_size': vocab_size, 'max_len': MAX_LEN, 'embed_dim': EMBED_DIM})

## Random search space (single Conv1D branch hyperparameters)

In [ ]:
param_distributions = {
    'num_filters'  : [32, 64, 96],        
    'kernel_size'  : [3, 5, 7],           
    'dense_units'  : [32, 64, 128],       
    'dropout'      : [0.2, 0.3, 0.4, 0.5], 
    'learning_rate': [1e-4, 3e-4, 5e-4],  
    'batch_size'   : [16, 32],
    'l2_reg'       : [0.0005, 0.001, 0.01]
}
N_ITER = 20  

random.seed(RANDOM_SEED)
sampled_configs = [
    {k: random.choice(v) for k, v in param_distributions.items()}
    for _ in range(N_ITER)
]

print(f'Total configs to try : {N_ITER}')
print('Sample config 1:', sampled_configs[0])

## Model, dataset, and training loop

Exact spec architecture: Embedding -> Conv1D (single branch) -> GlobalMaxPooling1D -> Dense -> Sigmoid(1). No gradient clipping (not needed — no recurrence).

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, lengths, y):
        self.X = torch.as_tensor(X, dtype=torch.long)
        self.lengths = torch.as_tensor(lengths, dtype=torch.long)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]


class SimpleCNNClassifier(nn.Module):
    """
    Embedding(FastText, trainable) -> Conv1D (single branch) -> ReLU
    -> masked GlobalMaxPooling1D -> Dense -> Sigmoid(1)
    """
    def __init__(self, vocab_size, embed_dim, embedding_matrix,
                 num_filters=256, kernel_size=3, dense_units=512, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = False

        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters,
                               kernel_size=kernel_size, padding=kernel_size // 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(num_filters, dense_units)
        self.fc2 = nn.Linear(dense_units, 1)

    def forward(self, x, lengths):
        B, L = x.shape
        emb = self.embedding(x).transpose(1, 2)   # (B, E, L) — Conv1d wants channels-first

        c = self.relu(self.conv(emb))              # (B, num_filters, L)

        # Masked GlobalMaxPooling1D — ignore padded positions
        arange = torch.arange(L, device=x.device).unsqueeze(0)
        mask = (arange < lengths.to(x.device).unsqueeze(1)).unsqueeze(1)  # (B, 1, L)
        c = c.masked_fill(~mask, float('-inf'))

        pooled = torch.max(c, dim=2).values         # GlobalMaxPooling1D -> (B, num_filters)

        h = self.dropout(pooled)
        h = self.relu(self.fc1(h))                   # Dense(512)
        h = self.dropout(h)
        logits = self.fc2(h).squeeze(-1)              # Dense(1), sigmoid via BCEWithLogitsLoss
        return logits


def build_model(config):
    return SimpleCNNClassifier(
        vocab_size, EMBED_DIM, embedding_matrix,
        num_filters = config['num_filters'],
        kernel_size = config['kernel_size'],
        dense_units = config['dense_units'],
        dropout     = config['dropout']
    ).to(DEVICE)


def train_model(model, X_tr, len_tr, y_tr, X_val, len_val, y_val, config,
                 epochs=30, patience=3, verbose=0):
    optimizer = torch.optim.Adam(
        (p for p in model.parameters() if p.requires_grad),
        lr=config['learning_rate'],
        weight_decay=config['l2_reg']
    )
    criterion = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(
        SequenceDataset(X_tr, len_tr, y_tr),
        batch_size=config['batch_size'], shuffle=True
    )
    X_val_t = torch.as_tensor(X_val, dtype=torch.long).to(DEVICE)
    len_val_t = torch.as_tensor(len_val, dtype=torch.long)
    y_val_t = torch.as_tensor(y_val, dtype=torch.float32).to(DEVICE)

    best_val_loss = float('inf')
    best_state = None
    patience_ctr = 0
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss, n_seen = 0.0, 0
        for xb, lb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb, lb)
            loss = criterion(logits, yb)
            loss.backward()
            # No gradient clipping — not needed for CNNs (no recurrence)
            optimizer.step()
            running_loss += loss.item() * len(xb)
            n_seen += len(xb)
        train_loss = running_loss / n_seen

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t, len_val_t)
            val_loss = criterion(val_logits, y_val_t).item()

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if verbose:
            print(f'  Epoch {epoch+1:2d}/{epochs} | loss: {train_loss:.4f} | val_loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, val_losses


def predict_proba(model, X, lengths, batch_size=256):
    model.eval()
    probs = []
    X_t = torch.as_tensor(X, dtype=torch.long)
    len_t = torch.as_tensor(lengths, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            xb = X_t[i:i+batch_size].to(DEVICE)
            lb = len_t[i:i+batch_size]
            logits = model(xb, lb)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)

## Random search - mean F1 across all 5 folds per config

For each sampled config: train/validate across all 5 `StratifiedGroupKFold` splits, average the F1 scores, then pick the config with the best mean F1 (matches the ML models' methodology).

In [ ]:
# Same sgkf as ML models — RANDOM_SEED=42 guarantees identical splits
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
splits = list(sgkf.split(X_train_seq, y_train, groups=groups))

print(f'Searching {N_ITER} configs across all {len(splits)} folds each...')

search_results = []

for i, config in enumerate(sampled_configs):
    fold_f1s = []

    for fold_num, (tr_idx, val_idx) in enumerate(splits):
        X_tr,   X_val   = X_train_seq[tr_idx], X_train_seq[val_idx]
        len_tr, len_val = len_train[tr_idx],   len_train[val_idx]
        y_tr,   y_val   = y_train[tr_idx],     y_train[val_idx]

        torch.manual_seed(RANDOM_SEED)
        model = build_model(config)

        train_model(
            model, X_tr, len_tr, y_tr, X_val, len_val, y_val, config,
            epochs=30, patience=3, verbose=0
        )

        y_proba_val = predict_proba(model, X_val, len_val)
        y_pred_val  = (y_proba_val > 0.5).astype(int)
        fold_f1     = f1_score(y_val, y_pred_val, average='weighted', zero_division=0)
        fold_f1s.append(fold_f1)

    mean_val_f1 = np.mean(fold_f1s)
    std_val_f1  = np.std(fold_f1s)

    search_results.append({
        'config': config,
        'val_f1': mean_val_f1,
        'val_f1_std': std_val_f1,
        'fold_f1s': fold_f1s
    })

    print(f'Config {i+1:2d}/{N_ITER} | mean_val_f1: {mean_val_f1:.4f} (+/- {std_val_f1:.4f}) | {config}')

    wandb.log({
        f'search/config_{i+1}_mean_val_f1': mean_val_f1,
        f'search/config_{i+1}_std_val_f1' : std_val_f1,
        **{f'search/config_{i+1}_{k}': v for k, v in config.items()}
    })

best_result = max(search_results, key=lambda x: x['val_f1'])
best_config = best_result['config']

print(f'\n✓ Random search complete.')
print(f'Best mean val F1 : {best_result["val_f1"]:.4f} (+/- {best_result["val_f1_std"]:.4f})')
print(f'Best config      : {best_config}')

wandb.config.update({
    'best_search_mean_val_f1': best_result['val_f1'],
    'best_search_std_val_f1' : best_result['val_f1_std'],
    **{f'best_param/{k}': v for k, v in best_config.items()}
})

## 5-fold cross-validation with best config

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy'   : accuracy_score(y_true, y_pred),
        'precision'  : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'     : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1'   : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auroc'      : roc_auc_score(y_true, y_proba)
    }

fold_metrics = []
all_histories = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    X_tr,   X_val   = X_train_seq[tr_idx],  X_train_seq[val_idx]
    len_tr, len_val = len_train[tr_idx],    len_train[val_idx]
    y_tr,   y_val   = y_train[tr_idx],      y_train[val_idx]

    torch.manual_seed(RANDOM_SEED)
    model = build_model(best_config)

    train_loss_hist, val_loss_hist = train_model(
        model, X_tr, len_tr, y_tr, X_val, len_val, y_val, best_config,
        epochs=30, patience=3, verbose=0
    )

    all_histories.append((train_loss_hist, val_loss_hist))

    y_proba = predict_proba(model, X_val, len_val)
    y_pred  = (y_proba > 0.5).astype(int)

    m = compute_metrics(y_val, y_pred, y_proba)
    m['fold'] = fold + 1
    fold_metrics.append(m)

    print(f"Fold {fold+1} | "
          f"Acc: {m['accuracy']:.4f} | "
          f"F1(w): {m['f1_weighted']:.4f} | "
          f"F1(macro): {m['f1']:.4f} | "
          f"AUROC: {m['auroc']:.4f}")

for fold, (train_loss, val_loss) in enumerate(all_histories):
    for epoch, (tl, vl) in enumerate(zip(train_loss, val_loss)):
        wandb.log({
            f'fold_{fold+1}/train_loss': tl,
            f'fold_{fold+1}/val_loss'  : vl,
            f'fold_{fold+1}/epoch'     : epoch
        })

fold_df    = pd.DataFrame(fold_metrics).set_index('fold')
mean_row   = fold_df.mean().rename('mean')
std_row    = fold_df.std().rename('std')
cv_summary = pd.concat([fold_df, mean_row.to_frame().T, std_row.to_frame().T])

print('\n=== CV Results (best config) ===')
print(cv_summary.round(4))

wandb.log({
    'cv/accuracy_mean'    : fold_df['accuracy'].mean(),
    'cv/accuracy_std'     : fold_df['accuracy'].std(),
    'cv/f1_weighted_mean' : fold_df['f1_weighted'].mean(),
    'cv/f1_weighted_std'  : fold_df['f1_weighted'].std(),
    'cv/f1_macro_mean'    : fold_df['f1'].mean(),
    'cv/f1_macro_std'     : fold_df['f1'].std(),
    'cv/auroc_mean'       : fold_df['auroc'].mean(),
    'cv/auroc_std'        : fold_df['auroc'].std(),
    'cv_results'          : wandb.Table(dataframe=cv_summary.round(4))
})

## Final holdout test evaluation

In [ ]:
X_tr_full, X_val_full, len_tr_full, len_val_full, y_tr_full, y_val_full = train_test_split(
    X_train_seq, len_train, y_train,
    test_size    = 0.1,
    stratify     = y_train,
    random_state = RANDOM_SEED
)

torch.manual_seed(RANDOM_SEED)
final_model = build_model(best_config)

train_model(
    final_model, X_tr_full, len_tr_full, y_tr_full, X_val_full, len_val_full, y_val_full, best_config,
    epochs=30, patience=3, verbose=1
)

y_proba_test = predict_proba(final_model, X_test_seq, len_test)
y_pred_test  = (y_proba_test > 0.5).astype(int)

test_metrics = compute_metrics(y_test, y_pred_test, y_proba_test)
np.save('y_pred-ft-1d-cnn.npy', y_pred_test)


print('=== HOLDOUT TEST SET RESULTS ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

wandb.log({f'test/{k}': v for k, v in test_metrics.items()})

report_df = pd.DataFrame(
    classification_report(y_test, y_pred_test,
                          target_names=le.classes_, output_dict=True)
).transpose().round(4)
wandb.log({'classification_report': wandb.Table(dataframe=report_df)})

In [ ]:
metrics_order = ['accuracy', 'precision', 'recall', 'f1_weighted', 'f1', 'auroc']

summary = pd.DataFrame({
    'CV Mean' : fold_df[metrics_order].mean().round(4),
    'CV Std'  : fold_df[metrics_order].std().round(4),
    'Test'    : pd.Series(test_metrics)[metrics_order].round(4)
})

print('=== PAPER TABLE — SimpleCNN (single Conv1D branch, FastText, PyTorch) ===')
print(summary)
print('\nBest hyperparameters:')
for k, v in best_config.items():
    print(f'  {k}: {v}')

wandb.log({'paper_table': wandb.Table(dataframe=summary)})
wandb.finish()